# Exercice 4.1 : quel modèle pour HelpDesk-Ops ? (45 min, J4 ≈ 11h00–12h00)

> 🗓️ **Session en 4 jours, salle GPU :** en séance, `granite4.1:3b` et `granite4.1:8b` (mode vLLM « agents ») ; `gemma4:12b` si le temps le permet (la bascule en mode juge prend 1 à 3 min). Sur le poste CPU de préparation, les trois modèles prennent ~40 min.

> Prérequis : J4-05. Notebook **autonome**. Sur le poste CPU de préparation, les trois modèles × deux experiments prennent ~40 minutes : lancez le calcul, puis préparez votre analyse pendant qu'il tourne.

**Question du responsable :** « Quel modèle mettre en production pour le routage et l'appel d'outils ? Je veux des chiffres. »

| Étape | Consigne |
|---|---|
| 1. Experiments | pour chaque modèle de `CANDIDATS` : `helpdesk-routage` (tâche `ev.tache_routage`, qui utilise par défaut la consigne v2 du J4-05 : on compare les **modèles**, à consigne égale) et `helpdesk-outils` (tâche `ev.tache_outil`), avec les évaluateurs du J4-05 |
| 2. Tableau | une ligne par modèle : `routage_juste`, `outil_valide`, `outil_attendu`, latence moyenne et p95 (`ev.resume`) |
| 3. Cas ratés | pour le meilleur modèle : quels cas restent faux ? lesquels figuraient déjà dans `origine` ? |
| 4. Langfuse | **Datasets → helpdesk-ops-eval → Runs** : sélectionnez les runs et comparez-les |
| 5. Recommandation | 5 lignes : le modèle choisi, **deux chiffres** qui le justifient, une limite, et ce qu'il faudrait mesurer en plus |

**Critères de réussite :** les six runs existent dans Langfuse · le tableau est produit par code (pas recopié à la main) · la recommandation cite des chiffres **et** une incertitude (20 cas : 1 cas = 5 points) · elle tient compte de la latence sur le matériel cible (GPU de la salle, pas CPU).

## 0. Mise en route et éléments fournis

In [ ]:
import os

from langfuse import observe, propagate_attributes

from helpdesk_ops.config import creer_llm
from helpdesk_ops.observabilite import init_observabilite
from helpdesk_ops.texte import sans_accents

langfuse, langfuse_handler = init_observabilite()   # .env chargé, masquage PII actif, clés vérifiées
llm = creer_llm("agent")                             # MODEL_AGENT, temperature=0, num_ctx=16384
print(f"✅ Langfuse : {os.environ['LANGFUSE_BASE_URL']} | modèle : {llm.model}")

In [ ]:
import time

import pandas as pd

from helpdesk_ops import evaluation as ev
from helpdesk_ops.config import basculer_vllm, creer_llm

pd.set_option("display.width", 180)
dataset = langfuse.get_dataset(ev.synchroniser_dataset(langfuse))

# Les trois candidats. gemma4 : raisonnement coupé (208 s contre 16 s par décision, mesuré en préparation)
CANDIDATS = {
    "granite4.1:3b": creer_llm("small"),
    "granite4.1:8b": creer_llm("agent"),
    "gemma4:12b": creer_llm("alt", reasoning=False),
}
# vLLM (salle) : sur 24 Go, gemma4 ne tient pas à côté des deux granite. Avant chaque modèle :
# basculer_vllm(MODE_VLLM[modele]) (1 à 3 min pour passer d'un mode à l'autre ; sans effet sous Ollama)
MODE_VLLM = {"granite4.1:3b": "agents", "granite4.1:8b": "agents", "gemma4:12b": "juge"}


def experience(nom: str, modele: str, tache, evaluateurs: list):
    debut = time.perf_counter()
    resultat = dataset.run_experiment(name=nom, run_name=f"{nom} · {modele}", task=tache, evaluators=evaluateurs,
                                      run_evaluators=[ev.ev_moyennes], max_concurrency=1, metadata={"modele": modele})
    print(f"   {nom:<17} {modele:<15} {time.perf_counter() - debut:5.0f} s")
    return resultat

## 1. Les experiments

In [ ]:
resultats = {}   # (experiment, modèle) -> résultat
for modele, candidat in CANDIDATS.items():
    basculer_vllm(MODE_VLLM[modele])   # fourni
    ...  # TODO : les deux experiments pour ce modèle
basculer_vllm("agents")                # fourni : retour aux modèles des autres notebooks

## 2. Le tableau comparatif

In [ ]:
# TODO : une ligne par modèle, colonnes routage_juste, outil_valide, outil_attendu, latences
tableau = ...

## 3. Les cas ratés du meilleur modèle

In [ ]:
# TODO : ev.lignes_resultat(...) filtré sur les échecs, avec la colonne origine

## 4. Votre recommandation

*(5 lignes : modèle, deux chiffres, une limite, la prochaine mesure.)*